List of evaluations

What needs to be measured?
- Data can capture the first order details
- Data can capture second order assocaition
- Data can capture low dimensional distribution,clusters
- Data can capture longitudinal trend
- Data can captuer missing pattern

#TODO: change learning rate for both synteg and halo
#TODO: get age, birth distribution map
#TODO: recover label from halo model

Basic Check:
- Softmax error
- Interval generation
- ART start, end, treatments
- Missing indicator

Utility:
* Marginal
    - total frequency (Use halo code) #Done
    - total prevalance (Use halo code) #Done
    - Weis.... divergent of continuous distribtuion #Done
    - perplexity # TODO: find function
    - disease label probability? Age of diagnosis code, gap to next recurrence (use synteg) # TODO: write function, get age, birth distribution
    - Column-wise correlation see benchmark #TODO: Find function, Use cosine similarity or Gower distance
    - latent clustering #UMAP used DONE #TODO: which distance to use
    - medical concept abundance
    - Canonical correlation
    - Latent analysis in synteg (use synteg)
* Longitudinal
    - bigram probability (Use halo code)
    - record length, mean
    - Sample Entropy?
    - Time-series similarity measures Dynamic Time Warping
    - Trend Analysis
    - Prediction model for some disease
    - MCMC likelihood?
* Missing Data
    - Missing proportion vs missing pattern #TODO: how to evaluate MNAR? MAR I can just compare P(R|X) of train vs syn? or AUC of P(R|X)? This is not testable under MAR or MNAR right?, Mariginal missingness
    - Parameter estimates
    - Missing indicator need to change
    - Missing visit proportions
* Patient Level consistancy: Any weird things, wrong trajectory. Ask pete for a list


Statistically:
- Auto-regressive plot
- logrank test of real vs synthetic survival

Privacy:
- Membership Inference
- Attribute disclosure


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import pickle
import itertools
from tqdm import tqdm
import matplotlib.pyplot as plt
from config import HALOConfig
from sklearn.metrics import r2_score
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import re
from scipy.stats import pearson3,wasserstein_distance
from sklearn.model_selection import train_test_split


In [ ]:
# %load ./HALO_Inpatient/evaluate_datasets.py
config = HALOConfig()

# label_mapping = pickle.load(open("./data/preprocess/idToLabel.pkl", "rb"))
label_mapping = [k for k in range(config.label_vocab_size)]
label_mapping.append('Overall')

train_ehr_dataset = pickle.load(open('./data/datasets/hiv/trainDataset.pkl', 'rb'))

# halo_ehr_dataset = pickle.load(open('./results/datasets/syn32_100bin_100Dataset.pkl', 'rb'))
# halo_ehr_dataset = pickle.load(open('./data/datasets/hiv/testDataset.pkl', 'rb'))
halo_ehr_dataset = pickle.load(open('./results/datasets/haloDataset.pkl', 'rb'))
halo_ehr_dataset = [{'labels': p['labels'], 'visits': p['visits']} for p in halo_ehr_dataset if len(p['visits']) > 0]
# !python synteg2halo.py -P "results/synteg/syntest4_45_150_250_0/" -S "./results/datasets/" -N "synteg45150250"
synteg_ehr_dataset = pickle.load(open('./results/datasets/haloDataset.pkl', 'rb'))
# synteg_ehr_dataset = pickle.load(open('./results/datasets/synteg_64_30_80_20Dataset.pkl', 'rb'))[:len(halo_ehr_dataset)]
synteg_ehr_dataset = [{'labels': p['labels'], 'visits': p['visits']} for p in synteg_ehr_dataset if len(p['visits']) > 0]


In [ ]:

stats = {}
for (l,d) in [('Train', train_ehr_dataset), ('HALO', halo_ehr_dataset), \
               ('SynTEG', synteg_ehr_dataset), \
                ]:
  aggregate_stats = {}
  label_probs = [len([p for p in d if p['labels'][i] == 1])/len(d) for i in range(25)]
  aggregate_stats['Labels'] = label_probs
  record_lens = [len(p['visits']) for p in d]
  visit_lens = [len(v) for p in d for v in p['visits']]
  aggregate_stats['Record Lengths'] = record_lens
  aggregate_stats['Visit Lengths'] = visit_lens
  stats[l] = aggregate_stats

pickle.dump(stats, open('results/shape.pkl', 'wb'))

def generate_statistics(ehr_dataset):
    stats = {}
    label_counts = {}
    for i in tqdm(range(config.label_vocab_size +1)):
        label_stats = {}
        d = [p for p in ehr_dataset if p['labels'][i] == 1] if i < config.label_vocab_size else ehr_dataset
        label_counts[label_mapping[i]] = len(d)


    for i in tqdm(range(config.label_vocab_size, config.label_vocab_size +1)):
        label_stats = {}
        d = [p for p in ehr_dataset if p['labels'][i] == 1] if i < config.label_vocab_size else ehr_dataset
        label_counts[label_mapping[i]] = len(d)

        aggregate_stats = {}
        record_lens = [len(p['visits']) for p in d]
        visit_lens = [len(v) for p in d for v in p['visits']]
        avg_record_len = np.mean(record_lens)
        std_record_len = np.std(record_lens)
        avg_visit_len = np.mean(visit_lens)
        std_visit_len = np.std(visit_lens)
        aggregate_stats["Record Length Mean"] = avg_record_len
        aggregate_stats["Record Length Standard Deviation"] = std_record_len
        aggregate_stats["Visit Length Mean"] = avg_visit_len
        aggregate_stats["Visit Length Standard Deviation"] = std_visit_len
        label_stats["Aggregate"] = aggregate_stats

        code_stats = {}
        n_records = len(record_lens)
        print(n_records)
        n_visits = len(visit_lens)
        record_code_counts = {}
        visit_code_counts = {}
        record_bigram_counts = {}
        visit_bigram_counts = {}
        record_sequential_bigram_counts = {}
        visit_sequential_bigram_counts = {}
        for p in d:
            patient_codes = set()
            patient_bigrams = set()
            sequential_bigrams = set()
            for j in range(len(p['visits'])):
                v = p['visits'][j]
                for c in v:
                    visit_code_counts[c] = 1 if c not in visit_code_counts else visit_code_counts[c] + 1
                    patient_codes.add(c)
                for cs in itertools.combinations(v,2):
                    cs = list(cs)
                    cs.sort()
                    cs = tuple(cs)
                    visit_bigram_counts[cs] = 1 if cs not in visit_bigram_counts else visit_bigram_counts[cs] + 1
                    patient_bigrams.add(cs)
                if j > 0:
                    v0 = p['visits'][j-1]
                    for c0 in v0:
                        for c in v:
                            sc = (c0, c)
                            visit_sequential_bigram_counts[sc] = 1 if sc not in visit_sequential_bigram_counts else visit_sequential_bigram_counts[sc] + 1
                            sequential_bigrams.add(sc)
            for c in patient_codes:
                record_code_counts[c] = 1 if c not in record_code_counts else record_code_counts[c] + 1
            for cs in patient_bigrams:
                record_bigram_counts[cs] = 1 if cs not in record_bigram_counts else record_bigram_counts[cs] + 1
            for sc in sequential_bigrams:
                record_sequential_bigram_counts[sc] = 1 if sc not in record_sequential_bigram_counts else record_sequential_bigram_counts[sc] + 1
        record_code_probs = {c: record_code_counts[c]/n_records for c in record_code_counts}
        visit_code_probs = {c: visit_code_counts[c]/n_visits for c in visit_code_counts}
        record_bigram_probs = {cs: record_bigram_counts[cs]/n_records for cs in record_bigram_counts}
        visit_bigram_probs = {cs: visit_bigram_counts[cs]/n_visits for cs in visit_bigram_counts}
        record_sequential_bigram_probs = {sc: record_sequential_bigram_counts[sc]/n_records for sc in record_sequential_bigram_counts}
        visit_sequential_bigram_probs = {sc: visit_sequential_bigram_counts[sc]/(n_visits - len(d)) for sc in visit_sequential_bigram_counts}
        code_stats["Per Record Code Probabilities"] = record_code_probs
        code_stats["Per Visit Code Probabilities"] = visit_code_probs
        code_stats["Per Record Bigram Probabilities"] = record_bigram_probs
        code_stats["Per Visit Bigram Probabilities"] = visit_bigram_probs
        code_stats["Per Record Sequential Visit Bigram Probabilities"] = record_sequential_bigram_probs
        code_stats["Per Visit Sequential Visit Bigram Probabilities"] = visit_sequential_bigram_probs
        label_stats["Probabilities"] = code_stats
        stats[label_mapping[i]] = label_stats
    label_probs = {l: label_counts[l]/n_records for l in label_counts}
    # label_probs = {l:0 for l in label_counts}
    stats["Label Probabilities"] = label_probs
    return stats
    
def generate_plots(stats1, stats2, label1, label2, types=["Per Record Code Probabilities", "Per Visit Code Probabilities", "Per Record Bigram Probabilities", "Per Visit Bigram Probabilities", "Per Record Sequential Visit Bigram Probabilities", "Per Visit Sequential Visit Bigram Probabilities"]):
    for i in tqdm(range(config.label_vocab_size, config.label_vocab_size +1)):
        print("\n")
        label = label_mapping[i]
        data1 = stats1[label]["Probabilities"]
        data2 = stats2[label]["Probabilities"]
        for j, t in enumerate(types):
            probs1 = data1[t]
            probs2 = data2[t]
            keys = set(probs1.keys()).union(set(probs2.keys()))
            # keys = [k for k in keys if k >= 30]
            values1 = [probs1[k] if k in probs1 else 0 for k in keys]
            values2 = [probs2[k] if k in probs2 else 0 for k in keys]
            r2score = r2_score(values1, values2)
            apd = np.sum(np.abs(np.subtract(values1, values2)))
            # print(f'{t}: {r2score}')

            plt.subplot(2,3,j+1)
            plt.clf()
            plt.scatter(values1, values2, marker=".", alpha=0.2)


            # Add the keys as text on the scatter plot
            # plt.scatter(values1, values2, marker=".", alpha=0.01)
            # for k, v1, v2 in zip(keys, values1, values2):
            #     plt.text(v1, v2, str(k), fontsize=5, alpha=0.75, ha='right')

            maxVal = min(1.1 * max(max(values1), max(values2)), 1.0)
            plt.xlim([0,maxVal])
            plt.ylim([0,maxVal])
            plt.title(f"{label} {t}")
            plt.text(0.3,0.95*maxVal,f'R-square= {r2score:3f}')
            plt.text(0.3,0.9*maxVal,f'APD = {apd:3f}')
            plt.plot([0,1], [0,1], 'k--', lw=2)
            plt.xlabel(label1)
            plt.ylabel(label2)
            plt.savefig(f"results/dataset_stats/plots/{label2}_{label.split(':')[-1]}_{t}_adjMax".replace(" ", "_"))
        
        # plt.show()

if True:
    ## Extract and save statistics
    train_ehr_stats = generate_statistics(train_ehr_dataset)
    halo_ehr_stats = generate_statistics(halo_ehr_dataset)
    synteg_ehr_stats = generate_statistics(synteg_ehr_dataset)
    pickle.dump(train_ehr_stats, open('results/dataset_stats/Train_Stats.pkl', 'wb'))
    pickle.dump(halo_ehr_stats, open('results/dataset_stats/HALO_Synthetic_Stats.pkl', 'wb'))
    pickle.dump(synteg_ehr_stats, open('results/dataset_stats/SynTEG_Synthetic_Stats.pkl', 'wb'))
    print(train_ehr_stats["Overall"]["Aggregate"])
    print(halo_ehr_stats["Overall"]["Aggregate"])
    print(synteg_ehr_stats["Overall"]["Aggregate"])
    print(train_ehr_stats["Label Probabilities"])
    print(halo_ehr_stats["Label Probabilities"])
    print(synteg_ehr_stats["Label Probabilities"])

    ## Plot per-code statistics
    generate_plots(train_ehr_stats, halo_ehr_stats, "Ccasanet Training Data", "HALO Synthetic Data")
    generate_plots(train_ehr_stats, synteg_ehr_stats, "CCasanet Training Data", "SynTEG Synthetic Data")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Define the paths to your images
image_paths = [
    "./results/dataset_stats/plots/HALO_Synthetic_Data_Overall_Per_Record_Code_Probabilities_adjMax.png",
    "./results/dataset_stats/plots/SynTEG_Synthetic_Data_Overall_Per_Record_Code_Probabilities_adjMax.png",
    "./results/dataset_stats/plots/HALO_Synthetic_Data_Overall_Per_Visit_Code_Probabilities_adjMax.png",
    "./results/dataset_stats/plots/SynTEG_Synthetic_Data_Overall_Per_Visit_Code_Probabilities_adjMax.png",
    "./results/dataset_stats/plots/HALO_Synthetic_Data_Overall_Per_Record_Sequential_Visit_Bigram_Probabilities_adjMax.png",
    "./results/dataset_stats/plots/SynTEG_Synthetic_Data_Overall_Per_Record_Sequential_Visit_Bigram_Probabilities_adjMax.png",
    "./results/dataset_stats/plots/HALO_Synthetic_Data_Overall_Per_Visit_Sequential_Visit_Bigram_Probabilities_adjMax.png",
    "./results/dataset_stats/plots/SynTEG_Synthetic_Data_Overall_Per_Visit_Sequential_Visit_Bigram_Probabilities_adjMax.png"
]

# Create a 4x2 grid of subplots
fig, axes = plt.subplots(4, 2, figsize=(16, 24))  # 16x24 inches for 4 rows and 2 columns

# Flatten the axes array for easy indexing
axes = axes.flatten()

# Loop over each subplot and image path to load and plot the image
for i, image_path in enumerate(image_paths):
    img = mpimg.imread(image_path)
    axes[i].imshow(img)
    axes[i].axis('off')  # Turn off axis for better visualization

# Adjust layout to ensure no overlaps
plt.tight_layout()
plt.show()


In [ ]:
prefix = "data/datasets/hiv/"
# age_map = np.load(prefix + 'age_map.npy',allow_pickle=True)  # new id starts from 0
# gender_map = np.load(prefix + 'gender_map.npy',allow_pickle=True)  # new id starts from 0
# country_map = np.load(prefix + 'country_map.npy',allow_pickle=True)  # new id starts from 0
# center_map = np.load(prefix + 'center_map.npy',allow_pickle=True)
# aidsmode_map = np.load(prefix + "mode_map.npy",allow_pickle=True)
# birth_map = np.load(prefix + "birth_map.npy",allow_pickle=True)

# group_to_id = list(age_map.tolist().keys()) + \
#             list(gender_map.tolist().keys()) + \
#             list(country_map.tolist().keys()) + \
#             list(center_map.tolist().keys()) + \
#             list(birth_map.tolist().keys()) + \
#             list(aidsmode_map.tolist().keys()) 

group_to_id = np.asanyarray(group_to_id)


# Determine the number of rows needed
prefix = "./data/datasets/hiv/"
code_map = np.load(prefix + "code_map.npy", allow_pickle=True).tolist()
interval_map = np.load("data/datasets/hiv/interval_map.npy",allow_pickle=True).tolist()
interval_map = [interval_map[x] for x in interval_map.keys()]
interval_map = {x:y for x,y in enumerate(interval_map)}

def halo2df(ehr_dataset):
    num_rows = sum(len(subject['visits']) for subject in ehr_dataset)

    # Preallocate a NumPy array for the one-hot encoding
    num_cols = len(code_map)
    one_hot_array = np.zeros((num_rows, num_cols), dtype=int)

    # Fill the NumPy array with one-hot encoded values
    row_idx = 0
    patient_ids = []
    intervals = []
    labels = pd.DataFrame(index=patient_ids,
                          columns=['age','gender','country','center','enroll','mode'])
    for id,subject in tqdm(enumerate(ehr_dataset),total = len(ehr_dataset)):
        labels.loc[id] = group_to_id[np.where(subject['labels']==1)]
        for episode in subject['visits']:
            # print(np.subtract(episode,30))
            indices = [index for index in np.subtract(episode,len(interval_map)) if -1 < index < len(code_map)]
            interval = [index for index in episode if -1 < index < len(interval_map)]
            one_hot_array[row_idx, indices] = 1
            intervals.append(interval[0])
            patient_ids.append(id)
            row_idx += 1

    # Convert the NumPy array to a DataFrame
    one_hot_df = pd.DataFrame(one_hot_array, columns=code_map.values())
    one_hot_df['patient_id'] = patient_ids

    ## Map interval
    one_hot_df['interval'] = intervals
    parameters = one_hot_df['interval'].map(interval_map)
    skewness = np.array([p[0] for p in parameters])
    skewness = np.where(np.isfinite(skewness), skewness, 0)
    mean = np.array([p[1] for p in parameters])
    std_dev = np.array([p[2] for p in parameters])
    lower = np.array([p[3] for p in parameters])
    upper =  np.array([p[4] for p in parameters])

    random_samples = pearson3.rvs(skew=skewness, loc=mean, scale=std_dev, size=len(skewness))
    # random_samples = parameters.apply(lambda x: np.random.choice(x[-1]) if isinstance(x, tuple) else np.nan)
    
    random_samples[random_samples<0] = 0.
    random_samples = np.round(random_samples)

    one_hot_df['interval'] = random_samples

    ## Map Age 
    ## TODO: change to use distribution
    labels.loc[:,'age'] = labels['age'].astype(float)
    labels.loc[:,'age'] = labels['age'].apply(lambda x : np.random.randint(x, x + 5))

    ## Map Enroll
    labels.loc[:,'enroll'] = labels['enroll'].astype(float)
    labels.loc[:,'enroll'] = labels['enroll'].apply(lambda x : np.random.randint(x, x + 5))

    labels = labels.reset_index().rename(columns={"index":"patient_id"})

    return one_hot_df, labels

print("converting training")
train_df,train_lb = halo2df(train_ehr_dataset)
print("converting HALO")
halo_df,halo_lb = halo2df(halo_ehr_dataset)
print("converting synteg")
synteg_df,synteg_lb = halo2df(synteg_ehr_dataset)

In [ ]:
import numpy as np
import pandas as pd
prefix = "data/datasets/hiv/"
age_map = np.load(prefix + 'age_map.npy',allow_pickle=True)  # new id starts from 0
gender_map = np.load(prefix + 'gender_map.npy',allow_pickle=True)  # new id starts from 0
country_map = np.load(prefix + 'country_map.npy',allow_pickle=True)  # new id starts from 0
center_map = np.load(prefix + 'center_map.npy',allow_pickle=True)
aidsmode_map = np.load(prefix + "mode_map.npy",allow_pickle=True)
birth_map = np.load(prefix + "birth_map.npy",allow_pickle=True)

def get_synteg(path):
    test_prefix = path

    # Load required data arrays
    code = np.load(test_prefix + 'code.npy', allow_pickle=True)
    age = np.load(test_prefix + 'age.npy', allow_pickle=True)
    gender = np.load(test_prefix + 'gender.npy', allow_pickle=True)
    interval = np.load(test_prefix + 'interval_con.npy', allow_pickle=True)
    length = np.load(test_prefix + 'length.npy', allow_pickle=True)
    country = np.load(test_prefix + 'country.npy', allow_pickle=True)
    center = np.load(test_prefix + 'center.npy', allow_pickle=True)
    aidsmode = np.load(test_prefix + 'mode.npy', allow_pickle=True)
    birth = np.load(test_prefix + 'birth.npy', allow_pickle=True)

    # Prepare subsets using list comprehension and slicing
    age_subset = [a[:l] for a, l in zip(age, length)]
    country_subset = [c[:l] for c, l in zip(country, length)]
    gender_subset = [g[:l] for g, l in zip(gender, length)]
    interval_subset = [i[:l] for i, l in zip(interval, length)]
    center_subset = [c[:l] for c, l in zip(center, length)]
    birth_subset = [b[:l] for b, l in zip(birth, length)]
    aidsmode_subset = [m[:l] for m, l in zip(aidsmode, length)]
    code_subset = [c[:l] for c, l in zip(code, length)]

    ## One-hot encoding

    # Load the code map and preallocate array
    code_map = np.load("data/datasets/hiv/code_map.npy", allow_pickle=True).tolist()
    num_rows = sum(len(subject) for subject in code_subset)
    num_cols = len(code_map)
    one_hot_array = np.zeros((num_rows, num_cols), dtype=int)
    patient_ids = np.empty(num_rows, dtype=int)  # Pre-allocate for efficiency

    # Fill one-hot array efficiently
    row_idx = 0
    for id, subject in enumerate(code_subset):
        for episode in subject:
            indices = np.intersect1d(episode, np.arange(num_cols))
            one_hot_array[row_idx, indices] = 1
            patient_ids[row_idx] = id
            row_idx += 1

    # Convert to DataFrame
    one_hot_df = pd.DataFrame(one_hot_array, columns=code_map.values())
    one_hot_df['patient_id'] = patient_ids

    ## Create test_labels in an efficient way

    # Generate rows directly using a list comprehension
    group_to_id = list(age_map.tolist().keys()) + \
            list(gender_map.tolist().keys()) + \
            list(country_map.tolist().keys()) + \
            list(center_map.tolist().keys()) + \
            list(birth_map.tolist().keys()) + \
            list(aidsmode_map.tolist().keys()) 
    
    rows = ({
        "patient_id": id,
        "age": a,
        "gender": g,
        "country": co,
        "center": ce,
        "birth": br,
        "aidsmode": ai,
        "interval": i
    } for id, (ages, genders, countries, centers, aidsmodes, births, intervals) in 
        enumerate(zip(age_subset, gender_subset, country_subset, center_subset, aidsmode_subset, birth_subset, interval_subset)) 
        for a, g, co, ce, ai, br, i in zip(ages, genders, countries, centers, aidsmodes, births, intervals))

    # Convert to DataFrame in one step
    labels = pd.DataFrame(rows)

    one_hot_df['interval'] = labels['interval']
    labels.drop(columns="interval",inplace = True)

    # ## TODO: change to use distribution
    labels['age'] = labels['age'].apply(lambda x : np.random.randint(x*5, x*5 + 5))
    # labels['age'] = labels.groupby('patient_id').age.min()

    ## Map Enroll
    labels['enroll'] = labels['birth'].apply(lambda x : np.random.randint(x, x + 5)) + 1955
    # labels['enroll'] = labels.groupby('patient_id').enroll.min()

    return one_hot_df, labels

if False:
    synteg_df, synteg_lb = get_synteg("results/synteg/syn32_51_65_60_0/")
    freq_train = train_df.drop(columns='interval').groupby('patient_id').max().mean(axis=0)
    freq_test = synteg_df.drop(columns='interval').groupby('patient_id').max().mean(axis=0)

    fig, ax = plt.subplots(figsize=(6, 5))

    ax.scatter(freq_train, freq_test,s=10)
    ax.set_xlabel('Frequency in Train Dataset')
    ax.set_ylabel('Frequency in Synthetic Dataset')
    ax.set_title('Scatter Plot: Frequency in Train vs Frequency in Synthetic')

    ax.plot(ax.get_xlim(), ax.get_ylim(), ls="--", c=".3", alpha=0.5)

    fig.show()
    

In [ ]:
## Basic Check
# print(synteg_lb.head())
### Softmax:
print(halo_df.filter(regex="cd4_v").sum(axis=1).value_counts())
print(synteg_df.filter(regex="cd4_v").sum(axis=1).value_counts())

### Softmax:
print(halo_df.filter(regex="rna_v").sum(axis=1).value_counts())
print(synteg_df.filter(regex="rna_v").sum(axis=1).value_counts())

### ART drugs


In [ ]:
def check_art(data):
    df_art = data.filter(regex="art|patient_id")
    df_art = df_art[df_art.filter(regex="art").sum(axis=1)>0]

    treatment_id = df_art.groupby("patient_id").art_sd.cumsum()
    is_treatment = df_art.groupby("patient_id").art_sd.cumsum() - df_art.groupby("patient_id").art_ed.cumsum()
    is_treatment[data.art_ed>0] = 1
    treatment_id[(is_treatment==0)] = np.nan

    ## Check SD match as ED
    art_grouped = df_art.groupby(['patient_id',treatment_id])
    start_end_match = (art_grouped.art_sd.sum() - art_grouped.art_ed.sum()).value_counts()

    ## Check drug consistency percentage
    drug_consistance = art_grouped.std().\
        filter(regex="art_id").mean()
    
    return start_end_match, drug_consistance

# print(check_art(train_df))
print(check_art(halo_df))
print(check_art(synteg_df))
### Defined Violation rate formula

In [ ]:
### Check set bundle

In [ ]:
train_ce = train_df.filter(regex='ce_id|patient_id').groupby('patient_id').max().mean()[1:]
train_ce = train_ce/train_ce.max()
halo_ce = halo_df.filter(regex='ce_id|patient_id').groupby('patient_id').max().mean()[1:]
halo_ce = halo_ce/halo_ce.max()
synteg_ce = synteg_df.filter(regex='ce_id|patient_id').groupby('patient_id').max().mean()[1:]
synteg_ce = synteg_ce/synteg_ce.max()

print("None-zero Clinical Event Train:",sum(train_ce>0))
print("Clincal Event's Distance: Train vs HALO",wasserstein_distance(train_ce,halo_ce))
print("None-zero Clinical Event HALO:",sum(halo_ce>0))
print("Clincal Event's Distance: Train vs SYNTEG",wasserstein_distance(train_ce,synteg_ce))
print("None-zero Clinical Event SYNTEG:",sum(synteg_ce>0))

df_ce = pd.DataFrame({
    "train": train_ce,
    "HALO": halo_ce,
    "SYNTEG": synteg_ce
})

with pd.option_context('display.max_rows', None):
    print(df_ce.sort_values(['SYNTEG','train']))


In [ ]:
# def calculate_ce_age(df, lb_df):
#     # Filter and merge with label data
#     ce_df = df.filter(regex="hivdiagnosis|enrol|art_sd|art_ed|ce_id|interval|patient_id")
#     ce_df = pd.merge(ce_df, lb_df[['patient_id', 'age']], on="patient_id")
    
#     # Calculate cumulative interval and adjust age
#     ce_df['interval'] = ce_df.groupby('patient_id').interval.cumsum() / 365
#     ce_df['age'] = ce_df.age + ce_df['interval']
    
#     # Calculate mean age of first onset for each condition event
#     first_age = {}
#     second_age = {}
#     for ce in ce_df.columns[:-3]:  # Loop through condition events excluding the last 3 columns
#         ce_onset = ce_df.loc[ce_df[ce] > 0, ['patient_id', 'age']]
#         first_onset = ce_onset.groupby('patient_id').age.first()
#         first_age[ce] = first_onset.dropna().values

#         second_onset = ce_onset.groupby('patient_id').age.diff()
#         second_age[ce] = second_onset.dropna().values
    
#     return first_age,second_age


# # Apply the function to each dataset
# train_first,train_second = calculate_ce_age(train_df, train_lb)
# halo_first,halo_second = calculate_ce_age(halo_df, halo_lb)
# synteg_first,synteg_second = calculate_ce_age(synteg_df, synteg_lb)



# # Plot
# fig, axes = plt.subplots(2,4,figsize = (12,8))

# def scplot(v1,v2,label,axes):
#     m1 = [np.mean(v1[x]) for x in v1.keys()]
#     m2 = [np.mean(v2[x]) for x in v2.keys()]

#     # Standard deviation calculation
#     std1 = [np.std(v1[x]) if len(v1[x]) > 0 else np.nan for x in v1.keys()]
#     std2 = [np.std(v2[x]) if len(v2[x]) > 0 else np.nan for x in v2.keys()]

#     # First plot (mean comparison)
#     sns.scatterplot(x=m1, y=m2, ax=axes[0])
#     min_limit = np.min([np.nanmin(m1), np.nanmin(m2)])
#     max_limit = np.max([np.nanmax(m1), np.nanmax(m2)])
#     axes[0].set_title(f"train v {label} mean")
#     axes[0].set_xlabel("train")
#     axes[0].set_xlim(min_limit, max_limit)
#     axes[0].set_ylabel(label)
#     axes[0].set_ylim(min_limit, max_limit)
#     axes[0].plot([min_limit, max_limit], [min_limit, max_limit], 'k--', lw=2)

#     # Second plot (std comparison)
#     sns.scatterplot(x=std1, y=std2, ax=axes[1])
#     min_limit_std = np.min([np.nanmin(std1), np.nanmin(std2)])
#     max_limit_std = np.max([np.nanmax(std1), np.nanmax(std2)])
#     axes[1].set_title(f"Train v {label} std")
#     axes[1].set_xlabel("Train")
#     axes[1].set_xlim(min_limit_std, max_limit_std)
#     axes[1].set_ylabel(label)
#     axes[1].set_ylim(min_limit_std, max_limit_std)
#     axes[1].plot([min_limit_std, max_limit_std], [min_limit_std, max_limit_std], 'k--', lw=2)


# scplot(train_first,halo_first,"HALO Ocurrent",axes=[axes[0,0],axes[0,2]])
# scplot(train_second,halo_second,"HALO Recurrence",axes=[axes[0,1],axes[0,3]])

# scplot(train_first,synteg_first,"SYNTEG Ocurrent",axes=[axes[1,0],axes[1,2]])
# scplot(train_second,synteg_second,"SYNTEG Recurrence",axes=[axes[1,1],axes[1,3]])

# fig.tight_layout()
# fig.show()

In [ ]:
### Load Unprocess Train Data
train_df_org = pd.read_csv("data/datasets/hiv/dt_code.csv.gz")
train_interval = pd.read_csv("data/datasets/hiv/dt_gap.csv.gz")
train_interval['interval'] = (train_interval.rename(columns={"diff":"interval"})["interval"]*365).astype(int)
train_interval.loc[train_interval.visit_index==1,'interval'] = 0
train_df_org = pd.concat([train_df_org,train_interval['interval']],axis=1)

cd4 = pd.read_csv("./data/imputed/lab_cd4.csv.gz")[['patient_id','cd4_v']].dropna()
rna = pd.read_csv("./data/imputed/lab_rna.csv.gz")[['patient_id','rna_v']].dropna()
visit = pd.read_csv("./data/imputed/visit.csv.gz")
weight = visit[['patient_id','weight']].dropna()
height = visit[['patient_id','height']].dropna()

idx,_ = train_test_split(np.unique(train_df_org.patient_id),test_size=0.15, random_state=0, shuffle=True)
cd4 = cd4[~cd4.patient_id.isin(idx)]
rna = rna[~rna.patient_id.isin(idx)]
weight = weight[~weight.patient_id.isin(idx)]
height = height[~height.patient_id.isin(idx)]

test_df = train_df_org[~train_df_org.patient_id.isin(idx)]
idx,_ = train_test_split(idx,test_size=0.1, random_state=0, shuffle=True)
train_df_org = train_df_org[train_df_org.patient_id.isin(idx)]

if False: pass
    # train_df_org['interval'] = np.digitize(train_df_org['interval'],[interval_map[x][4] for x in interval_map.keys()],right=True)

    # parameters = train_df_org['interval'].map(interval_map)
    # skewness = np.array([p[0] for p in parameters])
    # skewness = np.where(np.isfinite(skewness), skewness, 0)
    # mean = np.array([p[1] for p in parameters])
    # std_dev = np.array([p[2] for p in parameters])
    # lower = np.array([p[3] for p in parameters])
    # upper =  np.array([p[4] for p in parameters])

    # random_samples = pearson3.rvs(skew=skewness, loc=mean, scale=std_dev, size=len(skewness))
    # random_samples[random_samples<0] = 0.
    # random_samples = np.round(random_samples)

    # train_df_org['interval'] = random_samples

In [ ]:
## Missing Pattern comparison, Missing distribtion
stats = {}
missing_cols = train_df.filter(regex="_nan").columns

for col in missing_cols:
    stats[col] = {"train": train_df[col].mean(),
                  "halo": halo_df[col].mean(),
                  "synteg": synteg_df[col].mean()}
    
print(pd.DataFrame(stats))

## Missing Pattern comparison, Missing distribtion
fig, axes = plt.subplots(3,13,figsize = (16,8))
train_df.filter(regex="_nan").groupby(train_df.patient_id).mean().hist(bins=30,ax=axes[0])
halo_df.filter(regex="_nan").groupby(halo_df.patient_id).mean().hist(bins=30,ax=axes[1])
synteg_df.filter(regex="_nan").groupby(synteg_df.patient_id).mean().hist(bins=30,ax=axes[2])

fig.tight_layout()
fig.show()

In [ ]:
# ### Cosine similarity of marginal distribution

# import numpy as np
# import seaborn as sns
# import matplotlib.pyplot as plt
# from sklearn.metrics.pairwise import cosine_similarity
# from scipy.sparse import csr_matrix
# from scipy.sparse.linalg import norm

# def cosine_plot(data):
#     data = np.transpose(data)
#     data = csr_matrix(data)
#     cos_sim = cosine_similarity(data,dense_output=False)
    
#     return cos_sim

# train_corr = cosine_plot(train_df.drop(columns = "patient_id"))
# halo_corr = cosine_plot(halo_df.drop(columns = "patient_id"))
# synteg_corr = cosine_plot(synteg_df.drop(columns = "patient_id"))


# # Plot heatmap
# plt.figure(figsize=(4, 3))
# sns.heatmap(np.power(train_corr.toarray() - halo_corr.toarray(),2), annot=False, cmap='magma', square=True)
# plt.title(f"HALO's Cosine Similarity")
# plt.tight_layout()
# plt.savefig(f"results/dataset_stats/plots/HALO_cosine")
# plt.show()

# # Plot heatmap
# plt.figure(figsize=(4, 3))
# sns.heatmap(np.power(train_corr.toarray() - synteg_corr.toarray(),2), annot=False, cmap='magma', square=True)
# plt.title(f"SYNTEG's Cosine Similarity")
# plt.tight_layout()
# plt.savefig(f"results/dataset_stats/plots/SYNTEG_cosine")
# plt.show()

# stats = {}
# stats["Train v HALO"] = norm(train_corr - halo_corr)
# stats["Train v SYNTEG"] = norm(train_corr - synteg_corr)

# print("Frobenius Norm")
# print(stats)

In [ ]:
## Before comapring continuous distribution, Let's check if the distribution of bins matches



In [ ]:
## K-L divergence of the continuous variables

def RecoverContinuous(df):
    ## Continuous
    con_col = ["cd4_v","rna_v","weight","height"]
    df.columns = df.columns.str.replace("_v_below","_below")

    con_map = {f"{con}": np.load(f"data/datasets/hiv/{con.replace('_v', '')}_map.npy",allow_pickle=True).tolist() for con in con_col}

    df_con = pd.DataFrame()
    for con in con_col:
        con_val = df.filter(regex=con + ".+\\]$")
        con_val.columns = con_val.columns.str.replace(f"{con}_","")
        df_con[con] = con_val.idxmax(axis=1)
        parameters = df_con[con].map(con_map[con])

        # # Create arrays of skewness, mean, and std_dev from the mapped parameters
        # skewness = np.array([p[0] for p in parameters])
        # skewness = np.where(np.isfinite(skewness), skewness, 0)
        # mean = np.array([p[1] for p in parameters])
        # std_dev = np.array([p[2] for p in parameters])

        # # Generate all random samples using scipy.stats.pearson3.rvs in a single operation
        # # Using size=len(skewness) to generate one sample per row
        # random_samples = pearson3.rvs(skew=skewness, loc=mean, scale=std_dev, size=len(skewness))

        random_samples = parameters.apply(lambda x: np.random.choice(x[-1]) if isinstance(x, tuple) else np.nan)
        
        df_con[con] = random_samples
        df_con.loc[con_val.sum(axis=1)<1,con] = np.nan

    df_con.loc[df.rna_v_nan>0,"rna_v"] = np.nan
    df_con.loc[df.weight_nan>0,"weight"] = np.nan
    df_con.loc[df.height_nan>0,"height"] = np.nan

    return df_con

train_con = RecoverContinuous(train_df)
train_con["interval"] = train_df.interval

test_con = pd.DataFrame({"cd4_v":cd4.cd4_v,
                         "rna_v":rna.rna_v,
                         "weight":weight.weight,
                         "height":height.height})
test_con["interval"] = test_df.interval

halo_con = RecoverContinuous(halo_df)
halo_con["interval"] = halo_df.interval

synteg_con = RecoverContinuous(synteg_df)
synteg_con["interval"] = synteg_df.interval

# List of continuous variables to plot
continuous_vars = ["cd4_v", "rna_v", "weight", "height", "interval"]

# Plot each continuous variable as a separate density plot
plt.figure(figsize=(15, 10))
for i, var in enumerate(continuous_vars):
    plt.subplot(3, 2, i + 1)  # Adjusts grid for 5 plots
    sns.kdeplot(np.log(train_con[var].dropna() + 1), label="Train", fill=False, alpha=0.3)
    sns.kdeplot(np.log(test_con[var].dropna() + 1), label="Test", fill=False, alpha=0.3)
    sns.kdeplot(np.log(halo_con[var].dropna() + 1), label="Halo", fill=False, alpha=0.3)
    sns.kdeplot(np.log(synteg_con[var].dropna() + 1), label="Synteg", fill=False, alpha=0.3)
    
    plt.title(f'Density of {var}')
    plt.xlabel(f'Log-transformed {var}')
    plt.ylabel('Density')
    plt.legend()

plt.tight_layout()
plt.show()

print("Marginal Distance of continuous variables")
stats = {}
for col in continuous_vars:
    train2test = wasserstein_distance(train_con[col].dropna(), test_con[col].dropna())
    train2halo = wasserstein_distance(train_con[col].dropna(), halo_con[col].dropna())
    train2synteg = wasserstein_distance(train_con[col].dropna(), synteg_con[col].dropna())
    stats[col] = {"test": train2test, "halo": train2halo, "synteg": train2synteg}

# Display the marginal distances
print(pd.DataFrame(stats))

from scipy.stats import ks_2samp

print("KS test of continuous variables")
stats = {}
for col in continuous_vars:
    _,train2test = ks_2samp(train_con[col].dropna(), test_con[col].dropna())
    _,train2halo = ks_2samp(train_con[col].dropna(), halo_con[col].dropna())
    _,train2synteg = ks_2samp(train_con[col].dropna(), synteg_con[col].dropna())
    stats[col] = {"test": train2test, "halo": train2halo, "synteg": train2synteg}

# Display the marginal distances
print(pd.DataFrame(stats))

In [ ]:
halo_con.to_csv("./results/iwhod/halo_con.csv.gz",index=False,compression='gzip')
synteg_con.to_csv("./results/iwhod/synteg_con.csv.gz",index=False,compression='gzip')

In [ ]:
# Plot each continuous variable as a separate density plot
plt.figure(figsize=(15, 10))
for i, var in enumerate(continuous_vars):
    plt.subplot(3, 2, i + 1)  # Adjusts grid for 5 plots
    sns.kdeplot(np.log(train_con[var].dropna() + 1), label="Train", fill=False, alpha=0.3)
    sns.kdeplot(np.log(test_con[var].dropna() + 1), label="Test", fill=False, alpha=0.3)
    sns.kdeplot(np.log(halo_con[var].dropna() + 1), label="Halo", fill=False, alpha=0.3)
    # sns.kdeplot(np.log(synteg_con[var].dropna() + 1), label="Synteg", fill=False, alpha=0.3)
    
    plt.title(f'Density of {var}')
    plt.xlabel(f'Log-transformed {var}')
    plt.ylabel('Density')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
### Visit Length
train_con['patient_id'] = train_df.patient_id
train_con["visit_id"] = train_con.groupby("patient_id").cumcount()
train_con = train_con[train_con.visit_id>1]
train_con['time'] = train_con.groupby("patient_id").interval.cumsum()/365

test_con = test_df[['patient_id',"interval"]]
test_con["visit_id"] = test_con.groupby("patient_id").cumcount()
test_con = test_con[test_con.visit_id>1]
test_con['time'] = test_con.groupby("patient_id").interval.cumsum()/365

halo_con['patient_id'] = halo_df.patient_id
halo_con["visit_id"] = halo_con.groupby("patient_id").cumcount()
halo_con = halo_con[halo_con.visit_id>1]
halo_con['time'] = halo_con.groupby("patient_id").interval.cumsum()/365

synteg_con['patient_id'] = synteg_df.patient_id
synteg_con["visit_id"] = synteg_con.groupby("patient_id").cumcount()
synteg_con = synteg_con[synteg_con.visit_id>1]
synteg_con['time'] = synteg_con.groupby("patient_id").interval.cumsum()/365

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# Set up the figure
plt.figure(figsize=(10, 6))

# Define a function to calculate and plot density for each dataset
def plot_density(data, color, label):
    density = gaussian_kde(data)
    xs = np.linspace(min(data), max(data), 200)
    plt.plot(xs, density(xs), color=color, label=label, alpha=0.7)

# Prepare the data for each dataset
train_data = train_con.groupby("patient_id").visit_id.max().values
test_data = test_con.groupby("patient_id").visit_id.max().values
halo_data = halo_con.groupby("patient_id").visit_id.max().values
synteg_data = synteg_con.groupby("patient_id").visit_id.max().values

# Plot density lines for each dataset
plot_density(train_data, "blue", "Train")
plot_density(test_data, "orange", "Test")
plot_density(halo_data, "teal", "HALO")
plot_density(synteg_data, "magenta", "SYNTEG")

# Calculate and print means
train_mean = train_con.groupby("patient_id").visit_id.max().mean()
print("Mean max Visit ID - Train:", train_mean)

test_mean = test_con.groupby("patient_id").visit_id.max().mean()
print("Mean max Visit ID - Test:", test_mean)

halo_mean = halo_con.groupby("patient_id").visit_id.max().mean()
print("Mean max Visit ID - HALO:", halo_mean)

synteg_mean = synteg_con.groupby("patient_id").visit_id.max().mean()
print("Mean max Visit ID - SYNTEG:", synteg_mean)

# Customize the plot
plt.title("Density Plot of Max Visit ID per Patient")
plt.xlabel("Max Visit ID")
plt.ylabel("Density")
plt.legend()
plt.show()


In [ ]:
sns.lineplot(
    data=train_con,
    x="visit_id",
    y="time",
    estimator="mean",
    errorbar='sd',
    color="purple",
    lw=2,
    label="Train"
)

sns.lineplot(
    data=test_con,
    x="visit_id",
    y="time",
    estimator="mean",
    errorbar='sd',
    color="blue",
    lw=2,
    label="Test"
)

sns.lineplot(
    data=halo_con,
    x="visit_id",
    y="time",
    estimator="mean",
    errorbar='sd',
    color="red",
    lw=2,
    label="HALO"
)

sns.lineplot(
    data=synteg_con,
    x="visit_id",
    y="time",
    estimator="mean",
    errorbar='sd',
    color="green",
    lw=2,
    label="SYNTEG"
)

plt.title("Smooth Curve for Time")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import wasserstein_distance
from statsmodels.nonparametric.smoothers_lowess import lowess

# Define datasets for comparison
datasets = {
    'HALO': {'data': halo_con, 'color': 'red'},
    'SYNTEG': {'data': synteg_con, 'color': 'green'}
}

# Define the range of visit IDs
visit_ids = train_con['visit_id'].unique()

# Initialize dictionary to store Wasserstein distances
distances = {label: [] for label in datasets.keys()}

# Calculate Wasserstein distance for each visit_id
for visit_id in visit_ids:
    train_intervals = train_con[train_con['visit_id'] == visit_id]['interval']
    for label, info in datasets.items():
        comparison_data = info['data']
        comparison_intervals = comparison_data[comparison_data['visit_id'] == visit_id]['interval']
        
        # Compute Wasserstein distance and append to the distances dictionary
        if not train_intervals.empty and not comparison_intervals.empty:
            distance = wasserstein_distance(train_intervals, comparison_intervals)
            distances[label].append((visit_id, distance))

# Plot LOESS-smoothed curves of Wasserstein distance
plt.figure(figsize=(10, 6))

# Loop through each dataset to smooth and plot the Wasserstein distance
for label, data_points in distances.items():
    color = datasets[label]['color']
    visit_id_values, distance_values = zip(*data_points)  # Separate visit_ids and distances

    # Apply LOESS smoothing
    loess_result = lowess(endog=distance_values, exog=visit_id_values, frac=0.1)
    
    # Plot LOESS curve for Wasserstein distance
    plt.plot(loess_result[:, 0], loess_result[:, 1], color=color, lw=2, label=f"{label} vs Train (Wasserstein)")

# Customize plot
plt.title("LOESS Smoothed Curve of Wasserstein Distance for Time Intervals by Visit ID")
plt.xlabel("Visit ID")
plt.ylabel("Wasserstein Distance")
plt.legend()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.nonparametric.smoothers_lowess import lowess

# Define your datasets and colors for each group
datasets = {
    'Train': {'data': train_con, 'color': 'purple'},
    'HALO': {'data': halo_con, 'color': 'red'},
    'SYNTEG': {'data': synteg_con, 'color': 'green'}
}

# Plot smoothed LOESS curves with error bands
plt.figure(figsize=(10, 6))

# Loop through each dataset
for label, info in datasets.items():
    data = info['data']
    color = info['color']
    
    # Apply LOESS smoothing
    loess_result = lowess(endog=data['interval'], exog=data['visit_id'], frac=0.05)
    
    # Calculate residuals and standard deviation for error band
    residuals = data['interval'] - np.interp(data['visit_id'], loess_result[:, 0], loess_result[:, 1])
    std_dev = np.std(residuals)
    upper_bound = loess_result[:, 1] + std_dev
    lower_bound = loess_result[:, 1] - std_dev
    
    # Plot LOESS curve and error band
    plt.plot(loess_result[:, 0], loess_result[:, 1], color=color, lw=2, label=f"{label} (LOESS)")
    # plt.fill_between(loess_result[:, 0], lower_bound, upper_bound, color=color, alpha=0.1)

# Customize plot
plt.title("LOESS Smoothed Curve with Error Bands for Time between visits at each visit")
plt.xlabel("Visit ID")
plt.ylabel("Interval")
plt.legend()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Set the smoothing window
smooth_window = 5

# Define the datasets and their colors
datasets = {
    'Train': {'data': train_con, 'color': 'purple'},
    'HALO': {'data': halo_con, 'color': 'red'},
    'SYNTEG': {'data': synteg_con, 'color': 'green'}
}

# Plot the smoothed curves
plt.figure(figsize=(10, 6))

# Loop through each dataset
for label, info in datasets.items():
    data = info['data']
    color = info['color']
    
    # Apply rolling mean to smooth the curve
    smoothed_data = (
        data.groupby("visit_id")["interval"]
        .mean()
        .rolling(window=smooth_window, center=True)
        .mean()
        .reset_index()
    )
    
    # Plot the smoothed curve
    sns.lineplot(
        data=smoothed_data,
        x="visit_id",
        y="interval",
        color=color,
        lw=2,
        label=label
    )

# Customize plot
plt.title("Smoothed Curve for Time (Rolling Mean)")
plt.xlabel("Visit ID")
plt.ylabel("Interval")
plt.legend()
plt.show()


In [ ]:
# Define datasets and their labels in a dictionary
datasets = {
    "Train": train_df,
    "Test": test_df,
    "HALO": halo_df,
    "SYNTEG": synteg_df
}

# Loop through each dataset, calculate the mean, and print the result
for label, df in datasets.items():
    mean_follow_mode = df.filter(regex="follow_mode|patient_id").groupby("patient_id").max().mean()[1:]
    print(f"Mean Follow Mode - {label}:\n", mean_follow_mode)


In [ ]:
## Enrol-to-event

## Survival
surv_train = train_df[["patient_id","follow_mode_death","enrol_y","interval"]].copy()
surv_train['enrol_y'] = surv_train.groupby('patient_id').enrol_y.cumsum()
surv_train = surv_train[surv_train.enrol_y>0].groupby(['patient_id']).agg({'interval': 'sum', 'follow_mode_death': 'max'}).reset_index()
surv_train['type'] = 'train'

surv_test = test_df[["patient_id","follow_mode_death","enrol_y","interval"]].copy()
surv_test['enrol_y'] = surv_test.groupby('patient_id').enrol_y.cumsum()
surv_test = surv_test[surv_test.enrol_y>0].groupby(['patient_id']).agg({'interval': 'sum', 'follow_mode_death': 'max'}).reset_index()
surv_test['type'] = 'test'

surv_halo = halo_df[["patient_id","follow_mode_death","enrol_y","interval"]].copy()
surv_halo['enrol_y'] = surv_halo.groupby('patient_id').enrol_y.cumsum()
surv_halo = surv_halo[surv_halo.enrol_y>0].groupby(['patient_id']).agg({'interval': 'sum', 'follow_mode_death': 'max'}).reset_index()
surv_halo['type'] = 'halo'

surv_synteg = synteg_df[["patient_id","follow_mode_death","enrol_y","interval"]].copy()
surv_synteg['enrol_y'] = surv_synteg.groupby('patient_id').enrol_y.cumsum()
surv_synteg = surv_synteg[surv_synteg.enrol_y>0].groupby(['patient_id']).agg({'interval': 'sum', 'follow_mode_death': 'max'}).reset_index()
surv_synteg['type'] = 'synteg'

surv_df = pd.concat([surv_train,surv_test,surv_synteg],axis=0)
surv_df.columns = ['patient_id', 'time', 'event','type']


In [ ]:
surv_synteg.to_csv("./results/iwhod/synteg_surv.csv.gz",index=False,compression='gzip')

In [ ]:
%reload_ext rpy2.ipython

In [ ]:
%%R -i surv_df

library(survminer)
library(survival)
library(ggfortify)

## Log-rank test for train and test:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("train","test")),rho=0))

# ## Log-rank test for train and halo:
# print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("test","halo")),rho=0))

## Log-rank test for train and synteg:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("test","synteg")),rho=0))

survfit(Surv(time = surv_df$time/365, event = surv_df$event)~type,
             data = surv_df,
             conf.type = 'log-log') %>%
    ggsurvplot(fun = 'event',
            title = "time-to-death",
            conf.int = TRUE,
        #     xlim = c(0,50),
        #     ylim = c(0,0.6)
            )

In [ ]:
## Survival
surv_train = train_df[["patient_id","follow_mode_death","enrol_y","interval"]].copy()
surv_train['enrol_y'] = surv_train.groupby('patient_id').enrol_y.cumsum()
surv_train['cd4_v'] = train_con['cd4_v']
surv_train = surv_train[surv_train.enrol_y>0].groupby(['patient_id']).agg({'interval': 'sum',
                                                                                'cd4_v': 'mean',
                                                                                'follow_mode_death': 'max'}).reset_index()
surv_train = pd.merge(surv_train,train_lb,on="patient_id")

## Survival
surv_halo = halo_df[["patient_id","follow_mode_death","enrol_y","interval"]].copy()
surv_halo['enrol_y'] = surv_halo.groupby('patient_id').enrol_y.cumsum()
surv_halo['cd4_v'] = halo_con['cd4_v']
surv_halo = surv_halo[surv_halo.enrol_y>0].groupby(['patient_id']).agg({'interval': 'sum',
                                                                                'cd4_v': 'mean',
                                                                                'follow_mode_death': 'max'}).reset_index()
surv_halo = pd.merge(surv_halo,halo_lb,on="patient_id")

## Survival
surv_synteg = synteg_df[["patient_id","follow_mode_death","enrol_y","interval"]].copy()
surv_synteg['enrol_y'] = surv_synteg.groupby('patient_id').enrol_y.cumsum()
surv_synteg['cd4_v'] = synteg_con['cd4_v']
surv_synteg = surv_synteg[surv_synteg.enrol_y>0].groupby(['patient_id']).agg({'interval': 'sum',
                                                                                'cd4_v': 'mean',
                                                                                'follow_mode_death': 'max'}).reset_index()
surv_synteg = pd.merge(surv_synteg,synteg_lb,on="patient_id")

In [ ]:
%%R -i surv_train
library(rms)
library(tidyverse)

surv_train = filter(surv_train,gender !='8')

dd = datadist(surv_train)
dd$limits[c(1,3),'age'] = c(20,30)
dd$limits[c(1,3),'enroll'] = c(2010,2020)
dd$limits[c(1,3),'cd4_v'] = c(100,350)
dd$limits[c(1,3),'gender'] = c(0,1)
options(datadist = "dd",digits=3)

train_cox = cph(Surv(time = interval/365, event = follow_mode_death)~rcs(enroll,5) + rcs(age,5) + rcs(cd4_v,5) + gender + country,
    data = surv_train)
summary(train_cox)

In [ ]:
%%R -i surv_halo

halo_cox = cph(Surv(time = interval/365, event = follow_mode_death)~rcs(enroll,5) + rcs(age,5) + rcs(cd4_v,5) + gender + country,
    data = surv_halo)
summary(halo_cox,enroll,gender,age)

In [ ]:
%%R -i surv_synteg

dd$limits[c(1,3),'age'] = c(20,30)
dd$limits[c(1,3),'enroll'] = c(2010,2020)
dd$limits[c(1,3),'cd4_v'] = c(100,350)

synteg_cox = cph(Surv(time = interval/365, event = follow_mode_death)~rcs(enroll,5) + rcs(age,5) + gender + country,
    data = surv_synteg)
summary(synteg_cox,enroll,gender,age)

In [ ]:
import pandas as pd
from lifelines.statistics import logrank_test

# Function to prepare survival data for a given dataset and variable
def prepare_survival_data(df, variable):
    surv_df = df[["patient_id", variable, "enrol_y", "interval"]].copy()
    surv_df['enrol_y'] = surv_df.groupby('patient_id').enrol_y.cumsum()
    surv_df[f'{variable}_y'] = surv_df.groupby('patient_id')[variable].transform('max').gt(0).astype(int)
    surv_df['first_event'] = surv_df.groupby('patient_id')[variable].cumsum().eq(1)
    surv_df = surv_df[surv_df.groupby('patient_id')['first_event'].cumsum().eq(0) | surv_df['first_event']]
    surv_df = surv_df[surv_df.enrol_y > 0].drop(columns=[variable, 'first_event'])
    surv_df = surv_df.groupby(['patient_id']).agg({'interval': 'sum', f'{variable}_y': 'max'}).reset_index()
    surv_df.columns = ['patient_id', 'time', 'event']
    return surv_df

# Function to calculate log-rank p-values for a variable across groups
def calculate_logrank_pvalues(variable, train_df, test_df, halo_df, synteg_df):
    # Prepare survival data for all groups
    train_surv = prepare_survival_data(train_df, variable)
    test_surv = prepare_survival_data(test_df, variable)
    halo_surv = prepare_survival_data(halo_df, variable)
    synteg_surv = prepare_survival_data(synteg_df, variable)
    
    # Add group labels
    train_surv['type'] = 'train'
    test_surv['type'] = 'test'
    halo_surv['type'] = 'halo'
    synteg_surv['type'] = 'synteg'
    
    # Perform log-rank tests
    train_test = logrank_test(train_surv['time'], test_surv['time'], event_observed_A=train_surv['event'], event_observed_B=test_surv['event'])
    test_halo = logrank_test(test_surv['time'], halo_surv['time'], event_observed_A=test_surv['event'], event_observed_B=halo_surv['event'])
    test_synteg = logrank_test(test_surv['time'], synteg_surv['time'], event_observed_A=test_surv['event'], event_observed_B=synteg_surv['event'])
    
    return {
        'train_vs_test': train_test.p_value,
        'test_vs_halo': test_halo.p_value,
        'test_vs_synteg': test_synteg.p_value
    }

# Main function to process a list of variables
def logrank_analysis(variable_list, train_df, test_df, halo_df, synteg_df):
    results = []
    for variable in variable_list:
        p_values = calculate_logrank_pvalues(variable, train_df, test_df, halo_df, synteg_df)
        results.append({
            'Variable': variable,
            'Train vs Test': p_values['train_vs_test'],
            'Test vs Halo': p_values['test_vs_halo'],
            'Test vs Synteg': p_values['test_vs_synteg']
        })
    return pd.DataFrame(results)

# Example usage
variable_list = train_df.filter(regex='ce_id|art_sd|ce_is_ade|follow_mode_death').columns.to_list()  # Add more variables as needed
result_df = logrank_analysis(variable_list, train_df, test_df, halo_df, synteg_df)

# Display the results
# Function to calculate the false positive rate for each dataset
def calculate_false_positive_rate(df):
    # Create a new DataFrame to store the results
    false_positive_rates = {}
    
    for column in df.columns[1:]:  # Skip the first column ('Variable')
        p_values = df[column]
        false_positives = (p_values < 0.05).sum()
        false_positive_rate = false_positives / len(p_values)
        false_positive_rates[column] = false_positive_rate
    
    return false_positive_rates

# Calculate and display the false positive rates
false_positive_rates = calculate_false_positive_rate(result_df)
print(false_positive_rates)

with pd.option_context('display.max_rows', None):
    print(result_df)


In [ ]:
## Enrol-to-TB

## Survival
surv_train = train_df[["patient_id","ce_id_tuberculosis","enrol_y","interval"]].copy()
surv_train['enrol_y'] = surv_train.groupby('patient_id').enrol_y.cumsum()
surv_train['ce_id_tuberculosis_y'] = surv_train.groupby('patient_id')['ce_id_tuberculosis'].transform('max').gt(0).astype(int)
surv_train['first_tb_event'] = surv_train.groupby('patient_id')['ce_id_tuberculosis'].cumsum().eq(1)
surv_train = surv_train[surv_train.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_train['first_tb_event']]
surv_train = surv_train[surv_train.enrol_y>0].drop(columns=['ce_id_tuberculosis', 'first_tb_event'])
surv_train = surv_train.groupby(['patient_id']).agg({'interval': 'sum', 'ce_id_tuberculosis_y': 'max'}).reset_index()
surv_train['type'] = 'train'

surv_test = test_df[["patient_id","ce_id_tuberculosis","enrol_y","interval"]].copy()
surv_test['enrol_y'] = surv_test.groupby('patient_id').enrol_y.cumsum()
surv_test['ce_id_tuberculosis_y'] = surv_test.groupby('patient_id')['ce_id_tuberculosis'].transform('max').gt(0).astype(int)
surv_test['first_tb_event'] = surv_test.groupby('patient_id')['ce_id_tuberculosis'].cumsum().eq(1)
surv_test = surv_test[surv_test.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_test['first_tb_event']]
surv_test = surv_test[surv_test.enrol_y>0].drop(columns=['ce_id_tuberculosis', 'first_tb_event'])
surv_test = surv_test.groupby(['patient_id']).agg({'interval': 'sum', 'ce_id_tuberculosis_y': 'max'}).reset_index()
surv_test['type'] = 'test'

surv_halo = halo_df[["patient_id","ce_id_tuberculosis","enrol_y","interval"]].copy()
surv_halo['enrol_y'] = surv_halo.groupby('patient_id').enrol_y.cumsum()
surv_halo['ce_id_tuberculosis_y'] = surv_halo.groupby('patient_id')['ce_id_tuberculosis'].transform('max').gt(0).astype(int)
surv_halo['first_tb_event'] = surv_halo.groupby('patient_id')['ce_id_tuberculosis'].cumsum().eq(1)
surv_halo = surv_halo[surv_halo.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_halo['first_tb_event']]
surv_halo = surv_halo[surv_halo.enrol_y>0].drop(columns=['ce_id_tuberculosis', 'first_tb_event'])
surv_halo = surv_halo.groupby(['patient_id']).agg({'interval': 'sum', 'ce_id_tuberculosis_y': 'max'}).reset_index()
surv_halo['type'] = 'halo'

surv_synteg = synteg_df[["patient_id","ce_id_tuberculosis","enrol_y","interval"]].copy()
surv_synteg['enrol_y'] = surv_synteg.groupby('patient_id').enrol_y.cumsum()
surv_synteg['ce_id_tuberculosis_y'] = surv_synteg.groupby('patient_id')['ce_id_tuberculosis'].transform('max').gt(0).astype(int)
surv_synteg['first_tb_event'] = surv_synteg.groupby('patient_id')['ce_id_tuberculosis'].cumsum().eq(1)
surv_synteg = surv_synteg[surv_synteg.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_synteg['first_tb_event']]
surv_synteg = surv_synteg[surv_synteg.enrol_y>0].drop(columns=['ce_id_tuberculosis', 'first_tb_event'])
surv_synteg = surv_synteg.groupby(['patient_id']).agg({'interval': 'sum', 'ce_id_tuberculosis_y': 'max'}).reset_index()
surv_synteg['type'] = 'synteg'

surv_df = pd.concat([surv_train,surv_test,surv_halo,surv_synteg],axis=0)
surv_df.columns = ['patient_id', 'time', 'event','type']


In [ ]:
%reload_ext rpy2.ipython

In [ ]:
%%R -i surv_df

library(survminer)
library(survival)
library(ggfortify)

## Log-rank test for train and test:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("train","test")),rho=0))

## Log-rank test for train and halo:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("train","halo")),rho=0))

## Log-rank test for train and synteg:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("train","synteg")),rho=0))

survfit(Surv(time = surv_df$time/365, event = surv_df$event)~type,
             data = surv_df,
             subset = type != 'halo',
             conf.type = 'log-log') %>%
    ggsurvplot(fun = 'event',
            title = "time-to-TB",
            conf.int = TRUE,
        #     xlim = c(0,50),
        #     ylim = c(0,0.6)
            )

In [ ]:
## Enrol-to-art_sd

## Survival
surv_train = train_df[["patient_id","art_sd","enrol_y","interval"]].copy()
surv_train['enrol_y'] = surv_train.groupby('patient_id').enrol_y.cumsum()
surv_train['art_sd_y'] = surv_train.groupby('patient_id')['art_sd'].transform('max').gt(0).astype(int)
surv_train['first_art_sd_event'] = surv_train.groupby('patient_id')['art_sd'].cumsum().eq(1)
surv_train = surv_train[surv_train.groupby('patient_id')['first_art_sd_event'].cumsum().eq(0) | surv_train['first_art_sd_event']]
surv_train = surv_train[surv_train.enrol_y>0].drop(columns=['art_sd', 'first_art_sd_event'])
surv_train = surv_train.groupby(['patient_id']).agg({'interval': 'sum', 'art_sd_y': 'max'}).reset_index()
surv_train['type'] = 'train'

surv_test = test_df[["patient_id","art_sd","enrol_y","interval"]].copy()
surv_test['enrol_y'] = surv_test.groupby('patient_id').enrol_y.cumsum()
surv_test['art_sd_y'] = surv_test.groupby('patient_id')['art_sd'].transform('max').gt(0).astype(int)
surv_test['first_art_sd_event'] = surv_test.groupby('patient_id')['art_sd'].cumsum().eq(1)
surv_test = surv_test[surv_test.groupby('patient_id')['first_art_sd_event'].cumsum().eq(0) | surv_test['first_art_sd_event']]
surv_test = surv_test[surv_test.enrol_y>0].drop(columns=['art_sd', 'first_art_sd_event'])
surv_test = surv_test.groupby(['patient_id']).agg({'interval': 'sum', 'art_sd_y': 'max'}).reset_index()
surv_test['type'] = 'test'

surv_halo = halo_df[["patient_id","art_sd","enrol_y","interval"]].copy()
surv_halo['enrol_y'] = surv_halo.groupby('patient_id').enrol_y.cumsum()
surv_halo['art_sd_y'] = surv_halo.groupby('patient_id')['art_sd'].transform('max').gt(0).astype(int)
surv_halo['first_art_sd_event'] = surv_halo.groupby('patient_id')['art_sd'].cumsum().eq(1)
surv_halo = surv_halo[surv_halo.groupby('patient_id')['first_art_sd_event'].cumsum().eq(0) | surv_halo['first_art_sd_event']]
surv_halo = surv_halo[surv_halo.enrol_y>0].drop(columns=['art_sd', 'first_art_sd_event'])
surv_halo = surv_halo.groupby(['patient_id']).agg({'interval': 'sum', 'art_sd_y': 'max'}).reset_index()
surv_halo['type'] = 'halo'

surv_synteg = synteg_df[["patient_id","art_sd","enrol_y","interval"]].copy()
surv_synteg['enrol_y'] = surv_synteg.groupby('patient_id').enrol_y.cumsum()
surv_synteg['art_sd_y'] = surv_synteg.groupby('patient_id')['art_sd'].transform('max').gt(0).astype(int)
surv_synteg['first_art_sd_event'] = surv_synteg.groupby('patient_id')['art_sd'].cumsum().eq(1)
surv_synteg = surv_synteg[surv_synteg.groupby('patient_id')['first_art_sd_event'].cumsum().eq(0) | surv_synteg['first_art_sd_event']]
surv_synteg = surv_synteg[surv_synteg.enrol_y>0].drop(columns=['art_sd', 'first_art_sd_event'])
surv_synteg = surv_synteg.groupby(['patient_id']).agg({'interval': 'sum', 'art_sd_y': 'max'}).reset_index()
surv_synteg['type'] = 'synteg'

surv_df = pd.concat([surv_train,surv_test,surv_halo,surv_synteg],axis=0)
surv_df.columns = ['patient_id', 'time', 'event','type']


In [ ]:
%reload_ext rpy2.ipython

In [ ]:
%%R -i surv_df

library(survminer)
library(survival)
library(ggfortify)

## Log-rank test for train and test:
print(survdiff(Surv(time = time, event = event)~type,data = subset(surv_df,type %in% c("train","test")),rho=0))

## Log-rank test for train and halo:
print(survdiff(Surv(time = time, event = event)~type,data = subset(surv_df,type %in% c("train","halo")),rho=0))

## Log-rank test for train and synteg:
print(survdiff(Surv(time = time, event = event)~type,data = subset(surv_df,type %in% c("train","synteg")),rho=0))

survfit(Surv(time = log10(surv_df$time), event = surv_df$event)~type,
             data = surv_df,
             conf.type = 'log-log') %>%
    ggsurvplot(fun = 'event',
            title = "time-to-first-art",
            conf.int = TRUE,
            xlab = "Time(log10)"
        #     xlim = c(0,50),
        #     ylim = c(0,0.6)
            )

In [ ]:
## Enrol-to-AIDS

## Survival
surv_train = train_df[["patient_id","ce_is_ade","enrol_y","interval"]].copy()
surv_train['enrol_y'] = surv_train.groupby('patient_id').enrol_y.cumsum()
surv_train['ce_is_ade_y'] = surv_train.groupby('patient_id')['ce_is_ade'].transform('max').gt(0).astype(int)
surv_train['first_tb_event'] = surv_train.groupby('patient_id')['ce_is_ade'].cumsum().eq(1)
surv_train = surv_train[surv_train.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_train['first_tb_event']]
surv_train = surv_train[surv_train.enrol_y>0].drop(columns=['ce_is_ade', 'first_tb_event'])
surv_train = surv_train.groupby(['patient_id']).agg({'interval': 'sum', 'ce_is_ade_y': 'max'}).reset_index()
surv_train['type'] = 'train'

surv_test = test_df[["patient_id","ce_is_ade","enrol_y","interval"]].copy()
surv_test['enrol_y'] = surv_test.groupby('patient_id').enrol_y.cumsum()
surv_test['ce_is_ade_y'] = surv_test.groupby('patient_id')['ce_is_ade'].transform('max').gt(0).astype(int)
surv_test['first_tb_event'] = surv_test.groupby('patient_id')['ce_is_ade'].cumsum().eq(1)
surv_test = surv_test[surv_test.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_test['first_tb_event']]
surv_test = surv_test[surv_test.enrol_y>0].drop(columns=['ce_is_ade', 'first_tb_event'])
surv_test = surv_test.groupby(['patient_id']).agg({'interval': 'sum', 'ce_is_ade_y': 'max'}).reset_index()
surv_test['type'] = 'test'

surv_halo = halo_df[["patient_id","ce_is_ade","enrol_y","interval"]].copy()
surv_halo['enrol_y'] = surv_halo.groupby('patient_id').enrol_y.cumsum()
surv_halo['ce_is_ade_y'] = surv_halo.groupby('patient_id')['ce_is_ade'].transform('max').gt(0).astype(int)
surv_halo['first_tb_event'] = surv_halo.groupby('patient_id')['ce_is_ade'].cumsum().eq(1)
surv_halo = surv_halo[surv_halo.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_halo['first_tb_event']]
surv_halo = surv_halo[surv_halo.enrol_y>0].drop(columns=['ce_is_ade', 'first_tb_event'])
surv_halo = surv_halo.groupby(['patient_id']).agg({'interval': 'sum', 'ce_is_ade_y': 'max'}).reset_index()
surv_halo['type'] = 'halo'

surv_synteg = synteg_df[["patient_id","ce_is_ade","enrol_y","interval"]].copy()
surv_synteg['enrol_y'] = surv_synteg.groupby('patient_id').enrol_y.cumsum()
surv_synteg['ce_is_ade_y'] = surv_synteg.groupby('patient_id')['ce_is_ade'].transform('max').gt(0).astype(int)
surv_synteg['first_tb_event'] = surv_synteg.groupby('patient_id')['ce_is_ade'].cumsum().eq(1)
surv_synteg = surv_synteg[surv_synteg.groupby('patient_id')['first_tb_event'].cumsum().eq(0) | surv_synteg['first_tb_event']]
surv_synteg = surv_synteg[surv_synteg.enrol_y>0].drop(columns=['ce_is_ade', 'first_tb_event'])
surv_synteg = surv_synteg.groupby(['patient_id']).agg({'interval': 'sum', 'ce_is_ade_y': 'max'}).reset_index()
surv_synteg['type'] = 'synteg'

surv_df = pd.concat([surv_train,surv_test,surv_halo,surv_synteg],axis=0)
surv_df.columns = ['patient_id', 'time', 'event','type']


In [ ]:
%reload_ext rpy2.ipython

In [ ]:
%%R -i surv_df

library(survminer)
library(survival)
library(ggfortify)

## Log-rank test for train and test:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("train","test")),rho=0))

## Log-rank test for train and halo:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("train","halo")),rho=0))

## Log-rank test for train and synteg:
print(survdiff(Surv(time = time/365, event = event)~type,data = subset(surv_df,type %in% c("train","synteg")),rho=0))

survfit(Surv(time = surv_df$time/365, event = surv_df$event)~type,
             data = surv_df,
             conf.type = 'log-log') %>%
    ggsurvplot(fun = 'event',
            title = "time-to-AIDS",
            conf.int = TRUE,
        #     xlim = c(0,50),
        #     ylim = c(0,0.6)
            )

In [ ]:
import gc
gc.collect()

In [ ]:
import pandas as pd
import numpy as np

def long2wide(df, max_length):
    # Make a copy of the DataFrame and assign visit IDs
    df['visit_id'] = df.groupby('patient_id').cumcount() + 1
    
    # Truncate to max_length visits per patient
    df = df[df['visit_id'] <= max_length]
    
    # Find unique patients for indexing
    # unique_patients = df['patient_id'].unique()
    # Randomly sample 10,000 unique patient IDs
    unique_patients = np.random.choice(df['patient_id'].unique(), size=5000, replace=False)
    # Pre-allocate data for padding
    padding_rows = {
        'patient_id': np.repeat(unique_patients, max_length),
        'visit_id': np.tile(np.arange(1, max_length + 1), len(unique_patients))
    }
    
    # Prepare the padding DataFrame with zeros for other columns
    other_columns = [col for col in df.columns if col not in ['patient_id', 'visit_id']]
    for col in other_columns:
        padding_rows[col] = 0

    padding_df = pd.DataFrame(padding_rows)

    # Merge original data with padding and remove duplicates
    df = pd.concat([df, padding_df], ignore_index=True).drop_duplicates(
        subset=['patient_id', 'visit_id'], keep='first'
    )

    # Pivot to wide format
    df = df.pivot(index='patient_id', columns='visit_id', values=other_columns).fillna(0)

    # Flatten MultiIndex columns
    df.columns = [f'{col[0]}_visit{col[1]}' for col in df.columns]
    df = df.reset_index()

    return df

# Example usage:
# transformed_df = long2wide(df, max_length=10)


In [ ]:
train_df_wide = long2wide(train_df,max_length=265)
test_df_wide = long2wide(test_df,max_length=265)
halo_df_wide = long2wide(halo_df,max_length=265)
synteg_df_wide = long2wide(synteg_df,max_length=265)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Define the list of DataFrames in a more scalable way
data_frames = [train_df_wide, test_df_wide, halo_df_wide, synteg_df_wide]
labels = ["real", "test", "halo", "synteg"]

# To store the PCA-transformed data and eigenvalues for each dataset
scaler = StandardScaler()
pca_results = {}
explained_variances = {}

for i, df in enumerate(data_frames):
    # Drop 'patient_id' for PCA
    df_scaled = scaler.fit_transform(df.drop(columns="patient_id"))

    # Perform PCA with 30 components
    pca = PCA(n_components=30)
    df_pca = pca.fit_transform(df_scaled)
    pca_results[labels[i]] = df_pca
    explained_variances[labels[i]] = pca.explained_variance_[:30]  # Capture first 30 eigenvalues

# Set up a grid for plotting
fig, axes = plt.subplots(nrows=5, ncols=1, figsize=(10, 20))

# Plot density for each principal component for each dataset
for i, label in enumerate(labels):
    df_pca = pca_results[label]
    
    # Density plots for the first three principal components
    for j, pc in enumerate([0, 1, 2]):
        sns.kdeplot(df_pca[:, pc], ax=axes[i], fill=False, label=f'PC{j+1}', alpha=0.5)
        
    # Customize density plot
    axes[i].set_xlim(-200, 200)
    axes[i].set_xlabel('Principal Component Values')
    axes[i].set_ylabel('Density')
    axes[i].set_title(f'Density Plot of First 3 PCs for {label} data')
    axes[i].legend()

# Plot eigenvalues for the first 30 principal components in a single plot in the last row
for label, variances in explained_variances.items():
    axes[4].plot(range(1, 31), variances, marker='o', label=label, alpha=0.7)

# Customize the combined eigenvalue plot
axes[4].set_xlabel('Principal Components')
axes[4].set_ylabel('Eigenvalue')
axes[4].set_title('Eigenvalues of First 30 PCs for All Datasets')
axes[4].legend()

# Apply a tight layout and show the plot
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def combine_and_cluster_pca(data1, data2, k, label):
    # Standardize the data
    scaler = StandardScaler()
    
    # Ensure data is in numpy format
    data1 = scaler.fit_transform(np.array(data1))
    data2 = scaler.fit_transform(np.array(data2))

    # Concatenate the datasets
    combined_data = np.vstack((data1, data2))

    # Perform PCA to reduce to 2D
    pca = PCA(n_components=10)
    embedding = pca.fit_transform(combined_data)

    # Perform KMeans clustering on the 2D PCA embedding
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(embedding)

    # Create a label for source dataset (0 for data1, 1 for data2)
    source_labels = np.array([0] * len(data1) + [1] * len(data2))

    # Plot PCA result, colored by source (data1 or data2)
    plt.figure(figsize=(12, 6))
    
    # First subplot: Color by data source
    plt.subplot(1, 2, 1)
    plt.scatter(embedding[:, 0], embedding[:, 1], c=source_labels, cmap='coolwarm', s=10, alpha=0.6)
    plt.title(f"PCA (2D): Train vs {label}")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.colorbar(label=f"Data Source (0: Train, 1: {label})")

    # Second subplot: Histogram of cluster distribution for data1 and data2
    plt.subplot(1, 2, 2)

    # Cluster labels for data1 and data2
    data1_labels = labels[:len(data1)]
    data2_labels = labels[len(data1):]

    # Plot histograms of cluster distributions
    plt.hist([data1_labels, data2_labels], bins=np.arange(k+1) - 0.5, label=["Train", label], color=['blue', 'red'], 
             alpha=0.5,
             density=True, rwidth=0.85)
    plt.xticks(np.arange(k))
    plt.xlabel("Cluster")
    plt.ylabel("Number of points")
    plt.title(f"Cluster Distribution for Train and {label}")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"results/dataset_stats/plots/{label}_clustering_pca")
    plt.show()

# Example usage
combine_and_cluster_pca(train_df_wide[:5000], test_df_wide[:5000], k=3, label="Test")
combine_and_cluster_pca(train_df_wide[:5000], halo_df_wide[:5000], k=3, label="HALO")
combine_and_cluster_pca(train_df_wide[:5000], synteg_df_wide[:5000], k=3, label="SYNTEG")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def combine_and_cluster(data1, data2, k, label):

    # Standardize the data
    scaler = StandardScaler()
    
    # Ensure data is in numpy format
    data1 = scaler.fit_transform(np.array(data1))
    data2 = scaler.fit_transform(np.array(data2))

    # Concatenate the datasets
    combined_data = np.vstack((data1, data2))


    # Perform UMAP to reduce to 2D
    X = combined_data
    umap_reducer = umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=10, metric='cosine',n_jobs=-1)
    embedding = umap_reducer.fit_transform(X)

    # Use the first 2 components of the t-SNE embedding for visualization
    embedding_2d = embedding[:, :2]

    # Perform KMeans clustering on the full 10D t-SNE embedding
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(embedding)

    # Create a label for source dataset (0 for data1, 1 for data2)
    source_labels = np.array([0] * len(data1) + [1] * len(data2))

    # Plot t-SNE result, colored by source (data1 or data2)
    plt.figure(figsize=(12, 6))
    
    # First subplot: Color by data source
    plt.subplot(1, 2, 1)
    plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=source_labels, cmap='coolwarm', s=10,alpha=0.1)
    plt.title(f"UMAP (2D): Train vs {label}")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.colorbar(label=f"Data Source (0: Train, 1: {label})")

    # Second subplot: Histogram of cluster distribution for data1 and data2
    plt.subplot(1, 2, 2)

    # Cluster labels for data1 and data2
    data1_labels = labels[:len(data1)]
    data2_labels = labels[len(data1):]

    # Plot histograms of cluster distributions
    plt.hist([data1_labels, data2_labels], bins=np.arange(k+1) - 0.5, 
               label=["Train", label], color=['blue', 'red'], alpha=0.5,
               density=True, rwidth=0.85)
    plt.xticks(np.arange(k))
    plt.xlabel("Cluster")
    plt.ylabel("Number of points")
    plt.title(f"Cluster Distribution for Train and {label}")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"results/dataset_stats/plots/{label}_clustering")
    plt.show()

combine_and_cluster(train_df_wide[:5000],\
                   test_df_wide[:5000],\
                        k=3, label="Test")

combine_and_cluster(train_df_wide[:5000],\
                   halo_df_wide[:5000],\
                        k=3, label="HALO")

combine_and_cluster(train_df_wide[:5000],\
                   synteg_df_wide[:5000],\
                        k=3, label="SYNTEG")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

def combine_and_cluster_tsne(data1, data2, k, label):
    # Standardize the data
    scaler = StandardScaler()
    
    # Ensure data is in numpy format
    data1 = scaler.fit_transform(np.array(data1))
    data2 = scaler.fit_transform(np.array(data2))

    # Concatenate the datasets
    combined_data = np.vstack((data1, data2))

    # Perform t-SNE to reduce to 2D
    X = combined_data
    tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, n_iter=1000, random_state=42)
    embedding = tsne.fit_transform(X)

    # Use the first 2 components of the t-SNE embedding for visualization
    embedding_2d = embedding[:, :2]

    # Perform KMeans clustering on the 2D t-SNE embedding
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(embedding)

    # Create a label for source dataset (0 for data1, 1 for data2)
    source_labels = np.array([0] * len(data1) + [1] * len(data2))

    # Plot t-SNE result, colored by source (data1 or data2)
    plt.figure(figsize=(12, 6))
    
    # First subplot: Color by data source
    plt.subplot(1, 2, 1)
    plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=source_labels, cmap='coolwarm', s=10, alpha=0.6)
    plt.title(f"t-SNE (2D): Train vs {label}")
    plt.xlabel("t-SNE 1")
    plt.ylabel("t-SNE 2")
    plt.colorbar(label=f"Data Source (0: Train, 1: {label})")

    # Second subplot: Histogram of cluster distribution for data1 and data2
    plt.subplot(1, 2, 2)

    # Cluster labels for data1 and data2
    data1_labels = labels[:len(data1)]
    data2_labels = labels[len(data1):]

    # Plot histograms of cluster distributions
    plt.hist([data1_labels, data2_labels], bins=np.arange(k+1) - 0.5, 
            density=True,
            label=["Train", label], color=['blue', 'red'], 
            alpha=0.5, rwidth=0.85)
    plt.xticks(np.arange(k))
    plt.xlabel("Cluster")
    plt.ylabel("Number of points")
    plt.title(f"Cluster Distribution for Train and {label}")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"results/dataset_stats/plots/{label}_clustering_tsne")
    plt.show()

# Example usage
combine_and_cluster_tsne(train_df_wide[:5000],\
                         test_df_wide[:5000],\
                         k=3, label="Test")

combine_and_cluster_tsne(train_df_wide[:5000],\
                         halo_df_wide[:5000],\
                         k=3, label="HALO")

combine_and_cluster_tsne(train_df_wide[:5000],\
                         synteg_df_wide[:5000],\
                         k=3, label="SYNTEG")
